# 01 Convert / prepare MRI inputs

This notebook standardizes MRI inputs before FreeSurfer reconstruction. It is independent of the MEG preprocessing notebooks.

It supports mixed projects:

- some subjects may already have standardized MRI inputs such as `T1.mgz` or `*T1w*.nii.gz` under `paths.mri_root`
- other subjects may only have raw MRI exports/DICOM folders under `paths.mri_raw_root`

For each subject, the notebook first checks whether a standardized T1/T2 input already exists. If it does, conversion is skipped for that modality. If it does not, the notebook tries to create the standardized input from raw MRI data.

T1 is required for the standard `recon-all` workflow. T2 is optional and can later be used for `recon-all -T2 ... -T2pial` when configured.

Overwrite is controlled here with `OVERWRITE_STEPS`, not in the project config.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.workflow import existing_output_policy_for_step, should_overwrite
from meeg_pipeline.anatomy import (
    discover_raw_mri_subjects,
    mri_conversion_status_to_dataframe,
    prepare_anatomical_inputs_for_subjects,
    resolve_subjects,
    results_to_dataframe,
)

def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


In [ ]:
SUBJECTS = "all"
OVERWRITE_STEPS = []
DRY_RUN = False

conversion_policy = existing_output_policy_for_step("mri_conversion", OVERWRITE_STEPS)

pd.DataFrame([
    {
        "step": "mri_conversion",
        "overwrite": should_overwrite("mri_conversion", OVERWRITE_STEPS),
        "policy": conversion_policy,
    }
])

In [ ]:
selected_subjects = resolve_subjects(
    SUBJECTS,
    mri_raw_root=config.paths.mri_raw_root,
    mri_root=config.paths.mri_root,
    subjects_dir=config.freesurfer.subjects_dir,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
)

selected_subjects

In [ ]:
mri_conversion_status_to_dataframe(
    selected_subjects,
    mri_raw_root=config.paths.mri_raw_root,
    mri_root=config.paths.mri_root,
    t1_source_pattern=config.anatomy.conversion.t1_source_pattern,
    t2_source_pattern=config.anatomy.conversion.t2_source_pattern,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
)

In [ ]:
conversion_results = prepare_anatomical_inputs_for_subjects(
    selected_subjects,
    mri_raw_root=config.paths.mri_raw_root,
    mri_root=config.paths.mri_root,
    t1_source_pattern=config.anatomy.conversion.t1_source_pattern,
    t2_source_pattern=config.anatomy.conversion.t2_source_pattern,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
    freesurfer_home=config.freesurfer.home,
    make_mgz=config.anatomy.conversion.make_mgz,
    on_existing=conversion_policy,
    dry_run=DRY_RUN,
)

results_to_dataframe(conversion_results)

In [ ]:
mri_conversion_status_to_dataframe(
    selected_subjects,
    mri_raw_root=config.paths.mri_raw_root,
    mri_root=config.paths.mri_root,
    t1_source_pattern=config.anatomy.conversion.t1_source_pattern,
    t2_source_pattern=config.anatomy.conversion.t2_source_pattern,
    t1_patterns=config.anatomy.t1_patterns,
    t2_patterns=config.anatomy.t2_patterns,
)